In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# import pandas as pd
# data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
# data.to_csv("submission.csv", index=False)

**Import modules**

In [ ]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

DEBERTA_BASE = "microsoft/deberta-v3-small"
ROBERTA_BASE = "roberta-base"
DEBERTA_CKPT = "./deberta_finetuned"
ROBERTA_CKPT = "./roberta_finetuned"

OPTIONS = ["A", "B", "C", "D", "E"]
LABEL2ID = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

DEBERTA_W = 0.70
ROBERTA_W = 0.30





def map_at_k(true_answer, predictions, k=3):
    preds = predictions[:k]
    if true_answer in preds:
        return 1.0 / (preds.index(true_answer) + 1)
    return 0.0


def probs_to_top3(probs):
    order = np.argsort(probs)[::-1][:3]
    return [ID2LABEL[i] for i in order]


def top1_letter(probs):
    return ID2LABEL[int(np.argmax(probs))]


class MCQDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def get_tokenizer(model_name):
    if "deberta" in model_name.lower():
        from transformers import DebertaV2Tokenizer
        return DebertaV2Tokenizer.from_pretrained(model_name)
    return AutoTokenizer.from_pretrained(model_name)


def finetune_model(base_model, save_path, train_df):
    print(f"\nFine-tuning {base_model} -> {save_path}")

    tokenizer = get_tokenizer(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model,
        num_labels=5,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    texts = train_df["prompt"].astype(str).tolist()
    labels = [LABEL2ID[a] for a in train_df["answer"]]
    dataset = MCQDataset(texts, labels, tokenizer)

    args = TrainingArguments(
        output_dir=save_path + "_tmp",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        seed=42,
    )

    trainer = Trainer(model=model, args=args, train_dataset=dataset)
    trainer.train()

    os.makedirs(save_path, exist_ok=True)
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Saved checkpoint to {save_path}")


def load_or_train(base_model, save_path, train_df):
    if os.path.exists(os.path.join(save_path, "config.json")):
        print(f"Loading checkpoint from {save_path}")
        tokenizer = get_tokenizer(save_path) if "deberta" in save_path.lower() else AutoTokenizer.from_pretrained(save_path)
        model = AutoModelForSequenceClassification.from_pretrained(save_path)
    else:
        finetune_model(base_model, save_path, train_df)
        tokenizer = get_tokenizer(save_path) if "deberta" in save_path.lower() else AutoTokenizer.from_pretrained(save_path)
        model = AutoModelForSequenceClassification.from_pretrained(save_path)
    model.eval()
    return tokenizer, model

def predict_probs(tokenizer, model, text):
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt",
    )

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()[0]

    # safety against NaN values
    probs = np.nan_to_num(probs, nan=0.0, posinf=0.0, neginf=0.0)

    # renormalize
    s = probs.sum()
    if s == 0:
        probs = np.ones(5) / 5
    else:
        probs = probs / s

    return probs


# def predict_probs(tokenizer, model, text):
#     inputs = tokenizer(
#         text,
#         truncation=True,
#         padding=True,
#         max_length=256,
#         return_tensors="pt",
#     )
#     with torch.no_grad():
#         logits = model(**inputs).logits
#     return F.softmax(logits, dim=-1).numpy()[0]


# train = pd.read_csv(TRAIN_PATH)
# test = pd.read_csv(TEST_PATH)

# deb_tok, deb_model = load_or_train(DEBERTA_BASE, DEBERTA_CKPT, train)
# rob_tok, rob_model = load_or_train(ROBERTA_BASE, ROBERTA_CKPT, train)


def get_deberta_probs(text):
    return predict_probs(deb_tok, deb_model, text)


def get_roberta_probs(text):
    return predict_probs(rob_tok, rob_model, text)


def simple_ensemble(text):
    return (get_deberta_probs(text) + get_roberta_probs(text)) / 2


def weighted_ensemble(text):
    return DEBERTA_W * get_deberta_probs(text) + ROBERTA_W * get_roberta_probs(text)

#Q1

row25 = train.iloc[25]
prompt25 = str(row25["prompt"])
deb_probs_25 = get_deberta_probs(prompt25)

q1_letter = top1_letter(deb_probs_25)
q1_prob = round(float(deb_probs_25[LABEL2ID[q1_letter]]), 4)
q1_answer = f"{q1_letter}, {q1_prob}"
print(f"\nQ1 - DeBERTa top option (row 25): {q1_answer}")
print(f"ANSWER Q1: {q1_answer}")

#Q2

avg_probs_25 = simple_ensemble(prompt25)
q2_letter = top1_letter(avg_probs_25)
print(f"\nQ2 - Simple ensemble top option: {q2_letter}")
print(f"ANSWER Q2: {q2_letter}")

#Q3

w_probs_25 = weighted_ensemble(prompt25)
q3_letter = top1_letter(w_probs_25)
print(f"\nQ3 - Weighted ensemble top option: {q3_letter}")
print(f"ANSWER Q3: {q3_letter}")

#Q4

q4_answer = " ".join(probs_to_top3(w_probs_25))
print(f"\nQ4 - Top-3 prediction (row 25): {q4_answer}")
print(f"ANSWER Q4: {q4_answer}")

#Q5

sub_rows = []
for _, row in test.iterrows():
    prompt = str(row["prompt"])
    top3 = probs_to_top3(weighted_ensemble(prompt))
    sub_rows.append({"id": row["id"], "prediction": " ".join(top3)})

submission = pd.DataFrame(sub_rows)
submission.to_csv(SUBMISSION_PATH, index=False)

q5_answer = len(submission)
print(f"\nQ5 - Rows in submission.csv: {q5_answer}")
print(f"ANSWER Q5: {q5_answer}")

#Q6
TTA_PREFIX = "Please answer carefully:"
tta_changes = 0
for i in range(50):
    prompt = str(test.iloc[i]["prompt"])
    orig = top1_letter(get_deberta_probs(prompt))
    aug = top1_letter(get_deberta_probs(TTA_PREFIX + " " + prompt))
    tta_probs = (
        get_deberta_probs(prompt) + get_deberta_probs(TTA_PREFIX + " " + prompt)
    ) / 2
    tta_top1 = top1_letter(tta_probs)
    if tta_top1 != orig:
        tta_changes += 1

q6_answer = tta_changes
print(f"\nQ6 - TTA changed Top-1 count: {q6_answer}")
print(f"ANSWER Q6: {q6_answer}")

#Q7

diff_top1 = 0
for i in range(100):
    prompt = str(test.iloc[i]["prompt"])
    d1 = top1_letter(get_deberta_probs(prompt))
    e1 = top1_letter(weighted_ensemble(prompt))
    if d1 != e1:
        diff_top1 += 1

q7_answer = diff_top1
print(f"\nQ7 - Different Top-1 (DeBERTa vs Ensemble): {q7_answer}")
print(f"ANSWER Q7: {q7_answer}")

#Q8

pos_gain = 0
for i in range(100):
    prompt = str(test.iloc[i]["prompt"])
    deb = get_deberta_probs(prompt)
    ens = weighted_ensemble(prompt)
    deb_conf = float(np.max(deb))
    ens_conf = float(np.max(ens))
    if ens_conf - deb_conf > 0:
        pos_gain += 1

q8_answer = pos_gain
print(f"\nQ8 - Positive confidence gain count: {q8_answer}")
print(f"ANSWER Q8: {q8_answer}")

#Q9

top3_changes = 0
for i in range(100):
    prompt = str(test.iloc[i]["prompt"])
    deb_top3 = probs_to_top3(get_deberta_probs(prompt))
    ens_top3 = probs_to_top3(weighted_ensemble(prompt))
    if deb_top3 != ens_top3:
        top3_changes += 1

q9_answer = top3_changes
print(f"\nQ9 - Top-3 ranking changed count: {q9_answer}")
print(f"ANSWER Q9: {q9_answer}")

#Q10

map_scores = []
for i in range(100):
    row = train.iloc[i]
    prompt = str(row["prompt"])
    true_letter = row["answer"]
    pred_top3 = probs_to_top3(weighted_ensemble(prompt))
    map_scores.append(map_at_k(true_letter, pred_top3))

q10_answer = round(sum(map_scores) / len(map_scores), 4)
print(f"\nQ10 - MAP@3 (first 100 train rows): {q10_answer}")
print(f"ANSWER Q10: {q10_answer}")

submission = pd.DataFrame(sub_rows)
submission.to_csv(SUBMISSION_PATH, index=False)

print(f"Saved submission to: {SUBMISSION_PATH}")


In [ ]:
import pandas as pd
data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
data.to_csv("submission.csv", index=False)